### Imports and images

In [ ]:
import re
import time
import json
import ollama
from typing import Optional, List
from sina.config.paths import DATA
from pydantic import ValidationError, BaseModel, Field

store = "casa_ley"
city = "hermosillo"
date = "2026-02-26"
ley_path = DATA / store / city / date / "recortes"

class Product(BaseModel):
    name: str = Field(description="Nombre del producto tal como aparece en el flyer")
    brand: Optional[str] = Field(
        default=None,
        description="Marca del producto si tiene y es visible (ej: Bachoco, SuKarne, ETCO)"
    )
    price: Optional[float] = Field(
        default=None,
        description="Precio numérico del producto en oferta"
        )
    sale_type: Optional[str] = Field(
        default=None,
        description="Tipo de promoción aplicada al producto",
        examples=["precio_directo","2x1", "3x2", "2x$precio", "descuento"]
        )
    sale_description: Optional[str] = Field(
        default=None,
        description="Descripción literal de la oferta como aparece en el flyer",
        examples=["3x2 paga 2 y llévate 1 gratis", "30% de descuento", "a solo $49.90"]
    )
    unit: Optional[str] = Field(
        default=None,
        description="Unidad de venta del producto",
        examples=["kilo", "litro", "pieza", "en bolsa", "mazo"]
        )
    restrictions: Optional[str] = Field(description="Restricciones de la oferta en cuestión")

class FlyerExtraction(BaseModel):
    products: Optional[List[Product]] = Field(
        default=None,
        description="Lista de productos"
    )
    start_date: Optional[str] = Field(
        default=None,
        description="Fecha de inicio de precios y ofertas válidas (formato YYYY-MM-DD)"
        )
    end_date : Optional[str] = Field(
        default=None,
        description="Fecha de final de precios y ofertas válidas (formato YYYY-MM-DD)"
        )
    store: Optional[str] = Field(
        default=None,
        description="Nombre de la tienda (ej: casa_ley, walmart, abarrey, etc.)"
    )
    legal_warnings: Optional[str] = Field(
        default=None,
        description="Texto legal o restricciones generales de las ofertas"
    )
    extra_info: Optional[str] = Field(
        default=None,
        description="Cualquier información adicional relevante fuera de los productos o restricciones generales"
    )


In [2]:
flyer_schema = {
    "products": {
        "type": "list",
        "description": "Lista de productos",
        "items": {
            "name": {
                "type": "str",
                "required": True,
                "description": "Nombre del producto tal como aparece en el flyer"
            },
            "brand": {
                "type": "str",
                "required": False,
                "description": "Marca del producto si tiene y es visible (ej: Bachoco, SuKarne, ETCO)"
            },
            "price": {
                "type": "float",
                "required": False,
                "description": "Precio numérico del producto en oferta"
            },
            "sale_type": {
                "type": "str",
                "required": False,
                "description": "Tipo de promoción aplicada al producto",
                "examples": ["precio_directo", "2x1", "3x2", "2x$precio", "descuento"]
            },
            "sale_description": {
                "type": "str",
                "required": False,
                "description": "Descripción literal de la oferta como aparece en el flyer",
                "examples": ["3x2 paga 2 y llévate 1 gratis", "30% de descuento", "a solo $49.90"]
            },
            "unit": {
                "type": "str",
                "required": False,
                "description": "Unidad de venta del producto",
                "examples": ["kilo", "litro", "pieza", "en bolsa", "mazo"]
            },
            "restrictions": {
                "type": "str",
                "required": False,
                "description": "Restricciones de la oferta en cuestión"
            }
        }
    },
    "start_date": {
        "type": "str",
        "required": False,
        "description": "Fecha de inicio de precios y ofertas válidas (formato YYYY-MM-DD)"
    },
    "end_date": {
        "type": "str",
        "required": False,
        "description": "Fecha de final de precios y ofertas válidas (formato YYYY-MM-DD)"
    },
    "store": {
        "type": "str",
        "required": False,
        "description": "Nombre de la tienda (ej: casa_ley, walmart, abarrey, etc.)"
    },
    "legal_warnings": {
        "type": "str",
        "required": False,
        "description": "Texto legal o restricciones generales de las ofertas"
    },
    "extra_info": {
        "type": "str",
        "required": False,
        "description": "Cualquier información adicional relevante fuera de los productos o restricciones generales"
    }
}

In [3]:
imgs = []
for img_path in ley_path.iterdir(): 
    imgs.append(img_path) 

In [4]:
imgs

[WindowsPath('C:/Users/angel.merino/Documents/GitHub/sina/datos/casa_ley/hermosillo/2026-02-26/recortes/page_01_crop_000_frutas_verduras.jpg'),
 WindowsPath('C:/Users/angel.merino/Documents/GitHub/sina/datos/casa_ley/hermosillo/2026-02-26/recortes/page_01_crop_001_carnes.jpg'),
 WindowsPath('C:/Users/angel.merino/Documents/GitHub/sina/datos/casa_ley/hermosillo/2026-02-26/recortes/page_01_crop_002_abarrotes.jpg'),
 WindowsPath('C:/Users/angel.merino/Documents/GitHub/sina/datos/casa_ley/hermosillo/2026-02-26/recortes/page_01_crop_003_abarrotes.jpg'),
 WindowsPath('C:/Users/angel.merino/Documents/GitHub/sina/datos/casa_ley/hermosillo/2026-02-26/recortes/page_01_crop_004_ofertas_especiales.jpg'),
 WindowsPath('C:/Users/angel.merino/Documents/GitHub/sina/datos/casa_ley/hermosillo/2026-02-26/recortes/page_01_crop_005_otros.jpg'),
 WindowsPath('C:/Users/angel.merino/Documents/GitHub/sina/datos/casa_ley/hermosillo/2026-02-26/recortes/page_01_crop_006_banner.jpg'),
 WindowsPath('C:/Users/angel.

In [5]:
imgs[0]

WindowsPath('C:/Users/angel.merino/Documents/GitHub/sina/datos/casa_ley/hermosillo/2026-02-26/recortes/page_01_crop_000_frutas_verduras.jpg')

In [6]:
for img in imgs[:2]:
    print(img, img.exists(), img.stat().st_size)

C:\Users\angel.merino\Documents\GitHub\sina\datos\casa_ley\hermosillo\2026-02-26\recortes\page_01_crop_000_frutas_verduras.jpg True 405753
C:\Users\angel.merino\Documents\GitHub\sina\datos\casa_ley\hermosillo\2026-02-26\recortes\page_01_crop_001_carnes.jpg True 341303


In [7]:
from toon import encode
extract_text_prompt = {
    "rol": {
        "nombre": "Sina",
        "descripción": "eres un sistema experto en visión computacional y extracción de datos estructurados de flyers de supermercados."
    },
    "objetivo": "analizar las imágenes proporcionadas y extraer toda la información relevante de productos, precios, vigencias y avisos, estructurándola estrictamente en el formato JSON requerido",
    "reglas": {
        'fidelidad': 'extrae el texto tal cual aparece en la imagen. No inventes nombres de productos, marcas o precios',
        'informacion_parcial': 'estás evaluando recortes de un folleto más grande. si la imagen actual solo contiene productos y no menciona fechas de vigencia ni el nombre de la tienda, debes retornar esos campos como nulos (`null`)',
        'no_deduzcas': 'rellena el formato SOLO con la información visualmente presente en las imágenes de esta petición',
        'clasificacion_promos': 'presta especial atención a la mecánica de la oferta (ej. "3x2", "lleva 2 por $X", "descuento") y colócala en el campo correspondiente'
    },
    "formato_respuesta": {
        "instrucción": "responde ÚNICAMENTE con el objeto JSON. SIN markdown, SIN ```json, SIN explicaciones. Empieza directamente con { y termina con }",
        "campos_opcionales": "cualquier campo que NO sea visible en la imagen debe ser `null`. No todos los recortes contienen todos los campos — eso es esperado y correcto",
        "schema": f"{flyer_schema}"
    }
    
}

print(encode(extract_text_prompt))

rol:
  nombre: Sina
  descripción: eres un sistema experto en visión computacional y extracción de datos estructurados de flyers de supermercados.
objetivo: "analizar las imágenes proporcionadas y extraer toda la información relevante de productos, precios, vigencias y avisos, estructurándola estrictamente en el formato JSON requerido"
reglas:
  fidelidad: "extrae el texto tal cual aparece en la imagen. No inventes nombres de productos, marcas o precios"
  informacion_parcial: "estás evaluando recortes de un folleto más grande. si la imagen actual solo contiene productos y no menciona fechas de vigencia ni el nombre de la tienda, debes retornar esos campos como nulos (`null`)"
  no_deduzcas: rellena el formato SOLO con la información visualmente presente en las imágenes de esta petición
  clasificacion_promos: "presta especial atención a la mecánica de la oferta (ej. \"3x2\", \"lleva 2 por $X\", \"descuento\") y colócala en el campo correspondiente"
formato_respuesta:
  instrucción: "r

In [8]:
def clean_response(raw: str) -> dict:
    """Extrae el JSON de la respuesta, con o sin markdown wrapping."""
    clean = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw.strip())
    return json.loads(clean)

# Uso
# data = clean_response(gemma['message']['content'])
# result = FlyerExtraction.model_validate(data)

### local models

#### gemma3

##### prueba

In [30]:
# Test mínimo - sin system prompt, sin format, sin schema
response = ollama.chat(
    model="gemma3:27b",
    messages=[{
        'role': 'user',
        'content': '¿Qué ves en esta imagen?',
        'images': [str(imgs[0])]
    }]
)
print("Tokens:", response['prompt_eval_count'])
print(response['message']['content'][:500])

Tokens: 276
En la imagen veo un anuncio de precios de frutas y verduras en un supermercado. Aquí hay una lista de lo que veo, junto con sus precios:

*   **Chile Anaheim:** $49.90 por kilo.
*   **Mandarina:** $49.90 por kilo.
*   **Chile Serrano:** $39.90 por kilo.
*   **Cilantro:** $9.90 por mazos.
*   **Zanahoria A Granel:** $12.90 por kilo.
*   **Manzana Gala (bolsa de 600g):** $39.50
*   **Lechuga Romana:** $19.90 por pieza.
*   **Durazno Rojo:** $89.90 por kilo.
*   **Plátano Macho:** $39.90 por kilo.



##### c/ 1 imagen

In [12]:
messages = [
        {
            'role': 'system',
            'content': encode(extract_text_prompt)
        },
        {
            'role': 'user',
            'content': 'Analiza las siguientes imágenes y extrae la información',
            'images': [str(imgs[0])]
        }
    ]

In [13]:
start = time.perf_counter()
gemma = ollama.chat(
    model="gemma3:27b",
    messages=messages,
    options={'temperature': 0},
    format='json'
)

print(f"Duración: ", time.perf_counter() - start)

Duración:  462.23278010007925


In [16]:
dict(gemma)

{'model': 'gemma3:27b',
 'created_at': '2026-03-03T16:45:16.6012723Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 462191033800,
 'load_duration': 311139000,
 'prompt_eval_count': 1365,
 'prompt_eval_duration': 3905150500,
 'eval_count': 1747,
 'eval_duration': 456769358600,
 'message': Message(role='assistant', content='{\n  "products": [\n    {\n      "name": "CHILE Anahaim",\n      "brand": null,\n      "price": 49.90,\n      "sale_type": "precio_directo",\n      "sale_description": "a solo $49.90",\n      "unit": "Kilo",\n      "restrictions": "máximo 5 kilos por cliente"\n    },\n    {\n      "name": "MANDARINA",\n      "brand": null,\n      "price": 49.90,\n      "sale_type": "precio_directo",\n      "sale_description": "a solo $49.90",\n      "unit": "Kilo",\n      "restrictions": "máximo 5 kilos por cliente"\n    },\n    {\n      "name": "CHILE Serrano",\n      "brand": null,\n      "price": 39.90,\n      "sale_type": "precio_directo",\n      "sale_description": "

In [17]:
dict(dict(gemma)['message'])

{'role': 'assistant',
 'content': '{\n  "products": [\n    {\n      "name": "CHILE Anahaim",\n      "brand": null,\n      "price": 49.90,\n      "sale_type": "precio_directo",\n      "sale_description": "a solo $49.90",\n      "unit": "Kilo",\n      "restrictions": "máximo 5 kilos por cliente"\n    },\n    {\n      "name": "MANDARINA",\n      "brand": null,\n      "price": 49.90,\n      "sale_type": "precio_directo",\n      "sale_description": "a solo $49.90",\n      "unit": "Kilo",\n      "restrictions": "máximo 5 kilos por cliente"\n    },\n    {\n      "name": "CHILE Serrano",\n      "brand": null,\n      "price": 39.90,\n      "sale_type": "precio_directo",\n      "sale_description": "a solo $39.90",\n      "unit": "Kilo",\n      "restrictions": "máximo 5 kilos por cliente"\n    },\n    {\n      "name": "CILANTRO",\n      "brand": null,\n      "price": 9.90,\n      "sale_type": "precio_directo",\n      "sale_description": "a solo $9.90",\n      "unit": "Mazo",\n      "restrictions"

In [18]:
print(gemma.message.content)

{
  "products": [
    {
      "name": "CHILE Anahaim",
      "brand": null,
      "price": 49.90,
      "sale_type": "precio_directo",
      "sale_description": "a solo $49.90",
      "unit": "Kilo",
      "restrictions": "máximo 5 kilos por cliente"
    },
    {
      "name": "MANDARINA",
      "brand": null,
      "price": 49.90,
      "sale_type": "precio_directo",
      "sale_description": "a solo $49.90",
      "unit": "Kilo",
      "restrictions": "máximo 5 kilos por cliente"
    },
    {
      "name": "CHILE Serrano",
      "brand": null,
      "price": 39.90,
      "sale_type": "precio_directo",
      "sale_description": "a solo $39.90",
      "unit": "Kilo",
      "restrictions": "máximo 5 kilos por cliente"
    },
    {
      "name": "CILANTRO",
      "brand": null,
      "price": 9.90,
      "sale_type": "precio_directo",
      "sale_description": "a solo $9.90",
      "unit": "Mazo",
      "restrictions": "máximo 5 mazos por cliente"
    },
    {
      "name": "REPOLLO",
  

In [19]:
json.loads(gemma.message.content)

{'products': [{'name': 'CHILE Anahaim',
   'brand': None,
   'price': 49.9,
   'sale_type': 'precio_directo',
   'sale_description': 'a solo $49.90',
   'unit': 'Kilo',
   'restrictions': 'máximo 5 kilos por cliente'},
  {'name': 'MANDARINA',
   'brand': None,
   'price': 49.9,
   'sale_type': 'precio_directo',
   'sale_description': 'a solo $49.90',
   'unit': 'Kilo',
   'restrictions': 'máximo 5 kilos por cliente'},
  {'name': 'CHILE Serrano',
   'brand': None,
   'price': 39.9,
   'sale_type': 'precio_directo',
   'sale_description': 'a solo $39.90',
   'unit': 'Kilo',
   'restrictions': 'máximo 5 kilos por cliente'},
  {'name': 'CILANTRO',
   'brand': None,
   'price': 9.9,
   'sale_type': 'precio_directo',
   'sale_description': 'a solo $9.90',
   'unit': 'Mazo',
   'restrictions': 'máximo 5 mazos por cliente'},
  {'name': 'REPOLLO',
   'brand': None,
   'price': 14.9,
   'sale_type': 'precio_directo',
   'sale_description': 'a solo $14.90',
   'unit': 'Kilo',
   'restrictions': '

##### c/ batches

In [ ]:
batch_size = 2
flyer = {
    "products": [],
    "start_date": None,
    "end_date": None,
    "store": store,
    "legal_warnings": None,
    "extra_info": None
}

for i in range(0, len(imgs), batch_size):
    batch = imgs[i:i + batch_size]
    messages = [
            {
                'role': 'system',
                'content': encode(extract_text_prompt)
            },
            {
                'role': 'user',
                'content': f'Analiza las siguientes imágenes y extrae la información del supermercado {store}',
                'images': batch
            }
        ]
    
    try: 
        flyer_text = ollama.chat(
            model="gemma3:27b",
            messages=messages,
            options={'temperature': 0}
        )

        batch_data = json.loads(flyer_text.message.content)

        if batch_data.get("products"):
            flyer["products"].extend(batch_data["products"])
            
        for key in ["start_date", "end_date", "legal_warnings", "extra_info"]:
            new_value = batch_data.get(key)
            
            if new_value and not flyer.get(key):
                flyer[key] = new_value
            
            elif new_value and flyer.get(key) and key in ["legal_warnings", "extra_info"]:
                if new_value not in flyer[key]:
                    flyer[key] = f"{flyer[key]} | {new_value}"
                
    except json.JSONDecodeError as e:
        print(f"Error parseando JSON en el batch {i}: {e}")
    except Exception as e:
        print(f"Error procesando el batch {i}: {e}")

print(f"Total de productos extraídos: {len(flyer['products'])}")


Total de productos extraídos: 70


In [9]:
flyer

{'products': [{'name': 'Chile Anaheim',
   'restrictions': 'máximo 5 kilos por cliente',
   'price': 49.9,
   'unit': 'Kilo'},
  {'name': 'Mandarina',
   'restrictions': 'máximo 5 kilos por cliente',
   'price': 49.9,
   'unit': 'Kilo'},
  {'name': 'Chile Serrano',
   'restrictions': 'máximo 5 kilos por cliente',
   'price': 39.9,
   'unit': 'Kilo'},
  {'name': 'Cilantro',
   'restrictions': 'máximo 5 mazos por cliente',
   'price': 9.9,
   'unit': 'Mazo'},
  {'name': 'Repollo',
   'restrictions': 'máximo 5 kilos por cliente',
   'price': 14.9,
   'unit': 'Kilo'},
  {'name': 'Zanahoria A Granel',
   'restrictions': 'máximo 5 kilos por cliente',
   'price': 12.9,
   'unit': 'Kilo'},
  {'name': 'Manzana Gala',
   'restrictions': 'máximo 5 bolsas por cliente',
   'price': 39.5,
   'unit': 'bolsa de 600g'},
  {'name': 'Espinaca',
   'restrictions': 'máximo 5 mazos por cliente',
   'price': 9.9,
   'unit': 'Mazo'},
  {'name': 'Lechuga Romana',
   'restrictions': 'máximo 5 piezas por cliente

In [11]:
flyer['total_products'] = len(flyer['products'])
flyer

{'products': [{'name': 'Chile Anaheim',
   'restrictions': 'máximo 5 kilos por cliente',
   'price': 49.9,
   'unit': 'Kilo'},
  {'name': 'Mandarina',
   'restrictions': 'máximo 5 kilos por cliente',
   'price': 49.9,
   'unit': 'Kilo'},
  {'name': 'Chile Serrano',
   'restrictions': 'máximo 5 kilos por cliente',
   'price': 39.9,
   'unit': 'Kilo'},
  {'name': 'Cilantro',
   'restrictions': 'máximo 5 mazos por cliente',
   'price': 9.9,
   'unit': 'Mazo'},
  {'name': 'Repollo',
   'restrictions': 'máximo 5 kilos por cliente',
   'price': 14.9,
   'unit': 'Kilo'},
  {'name': 'Zanahoria A Granel',
   'restrictions': 'máximo 5 kilos por cliente',
   'price': 12.9,
   'unit': 'Kilo'},
  {'name': 'Manzana Gala',
   'restrictions': 'máximo 5 bolsas por cliente',
   'price': 39.5,
   'unit': 'bolsa de 600g'},
  {'name': 'Espinaca',
   'restrictions': 'máximo 5 mazos por cliente',
   'price': 9.9,
   'unit': 'Mazo'},
  {'name': 'Lechuga Romana',
   'restrictions': 'máximo 5 piezas por cliente

In [9]:
batch_size = 1
flyer = {
    "products": [],
    "start_date": None,
    "end_date": None,
    "store": store,
    "legal_warnings": None,
    "extra_info": None
}

for i in range(0, len(imgs), batch_size):
    batch = imgs[i:i + batch_size]
    batch_num = i // batch_size + 1
    messages = [
            {
                'role': 'system',
                'content': encode(extract_text_prompt)
            },
            {
                'role': 'user',
                'content': f'Analiza las siguientes imágenes y extrae la información del supermercado {store}',
                'images': batch
            }
        ]
    
    try: 
        print(f"\n⏳ Enviando batch {batch_num} a Ollama (Contiene {len(batch)} imágenes)...")
        start_time = time.time()
        flyer_text = ollama.chat(
            model = "gemma3:27b",
            messages = messages,
            options = {'temperature': 0}
        )

        elapsed_time = time.time() - start_time
        print(f"✅ Ollama respondió en {elapsed_time:.2f} segundos.")

        batch_data  = clean_response(flyer_text.message.content)

        print(batch_data)

        if batch_data.get("products"):
            flyer["products"].extend(batch_data["products"])
            print(f"  Batch {batch_num}: {len(batch_data['products'])} productos")
            
        for key in ["start_date", "end_date", "legal_warnings", "extra_info"]:
            new_value = batch_data.get(key)
            
            if new_value and not flyer.get(key):
                flyer[key] = new_value
            
            elif new_value and flyer.get(key) and key in ["legal_warnings", "extra_info"]:
                if new_value not in flyer[key]:
                    flyer[key] = f"{flyer[key]} | {new_value}"
            
    except json.JSONDecodeError as e:
        print(f"✗ Error parseando JSON en batch {batch_num}: {e}")
    except ValidationError as e:
        print(f"✗ Error de validación en batch {batch_num}: {e}")
    except Exception as e:
        print(f"✗ Error procesando batch {batch_num}: {e}")

flyer['total_products'] = len(flyer['products'])

print(f"\nTotal de productos extraídos: {len(flyer['products'])}")


⏳ Enviando batch 1 a Ollama (Contiene 1 imágenes)...
✅ Ollama respondió en 429.29 segundos.
{'products': [{'name': 'CHILE Anahiem', 'brand': None, 'price': 49.9, 'sale_type': 'precio_directo', 'sale_description': 'a solo $49.90', 'unit': 'Kilo', 'restrictions': 'máximo 5 kilos por cliente'}, {'name': 'MANDARINA', 'brand': None, 'price': 49.9, 'sale_type': 'precio_directo', 'sale_description': 'a solo $49.90', 'unit': None, 'restrictions': 'máximo 5 kilos por cliente'}, {'name': 'CHILE Serrano', 'brand': None, 'price': 39.9, 'sale_type': 'precio_directo', 'sale_description': 'a solo $39.90', 'unit': 'Kilo', 'restrictions': 'máximo 5 kilos por cliente'}, {'name': 'CILANTRO', 'brand': None, 'price': 9.9, 'sale_type': 'precio_directo', 'sale_description': 'a solo $9.90', 'unit': 'Mazo', 'restrictions': 'máximo 5 mazos por cliente'}, {'name': 'REPOLLO', 'brand': None, 'price': 14.9, 'sale_type': 'precio_directo', 'sale_description': 'a solo $14.90', 'unit': 'Kilo', 'restrictions': 'máximo 

In [10]:
flyer

{'products': [{'name': 'CHILE Anahiem',
   'brand': None,
   'price': 49.9,
   'sale_type': 'precio_directo',
   'sale_description': 'a solo $49.90',
   'unit': 'Kilo',
   'restrictions': 'máximo 5 kilos por cliente'},
  {'name': 'MANDARINA',
   'brand': None,
   'price': 49.9,
   'sale_type': 'precio_directo',
   'sale_description': 'a solo $49.90',
   'unit': None,
   'restrictions': 'máximo 5 kilos por cliente'},
  {'name': 'CHILE Serrano',
   'brand': None,
   'price': 39.9,
   'sale_type': 'precio_directo',
   'sale_description': 'a solo $39.90',
   'unit': 'Kilo',
   'restrictions': 'máximo 5 kilos por cliente'},
  {'name': 'CILANTRO',
   'brand': None,
   'price': 9.9,
   'sale_type': 'precio_directo',
   'sale_description': 'a solo $9.90',
   'unit': 'Mazo',
   'restrictions': 'máximo 5 mazos por cliente'},
  {'name': 'REPOLLO',
   'brand': None,
   'price': 14.9,
   'sale_type': 'precio_directo',
   'sale_description': 'a solo $14.90',
   'unit': 'Kilo',
   'restrictions': 'má

In [ ]:
path = DATA / store / city / date / "gemma_flyer_data.json"

with open('gemma3.json', 'w', encoding='utf-8') as f:
    json.dump(flyer, f, ensure_ascii=False, indent= 3)

#### qwen

In [ ]:
batch_size = 1
flyer = {
    "products": [],
    "start_date": None,
    "end_date": None,
    "store": store,
    "legal_warnings": None,
    "extra_info": None
}

for i in range(0, len(imgs), batch_size):
    batch = imgs[i:i + batch_size]
    batch_num = i // batch_size + 1
    messages = [
            {
                'role': 'system',
                'content': encode(extract_text_prompt)
            },
            {
                'role': 'user',
                'content': f'Analiza las siguientes imágenes y extrae la información del supermercado {store}',
                'images': batch
            }
        ]
    
    try: 
        print(f"\n⏳ Enviando batch {batch_num} a Ollama (Contiene {len(batch)} imágenes)...")
        start_time = time.time()
        flyer_text = ollama.chat(
            model = "qwen3-vl:30b",
            messages = messages,
            options = {'temperature': 0}
        )

        elapsed_time = time.time() - start_time
        print(f"✅ Ollama respondió en {elapsed_time:.2f} segundos.")

        batch_data  = clean_response(flyer_text.message.content)

        print(batch_data)

        if batch_data.get("products"):
            flyer["products"].extend(batch_data["products"])
            print(f"  Batch {batch_num}: {len(batch_data['products'])} productos")
            
        for key in ["start_date", "end_date", "legal_warnings", "extra_info"]:
            new_value = batch_data.get(key)
            
            if new_value and not flyer.get(key):
                flyer[key] = new_value
            
            elif new_value and flyer.get(key) and key in ["legal_warnings", "extra_info"]:
                if new_value not in flyer[key]:
                    flyer[key] = f"{flyer[key]} | {new_value}"
            
    except json.JSONDecodeError as e:
        print(f"✗ Error parseando JSON en batch {batch_num}: {e}")
    except ValidationError as e:
        print(f"✗ Error de validación en batch {batch_num}: {e}")
    except Exception as e:
        print(f"✗ Error procesando batch {batch_num}: {e}")

flyer['total_products'] = len(flyer['products'])

print(f"\nTotal de productos extraídos: {len(flyer['products'])}")


⏳ Enviando batch 1 a Ollama (Contiene 1 imágenes)...
✅ Ollama respondió en 425.12 segundos.
{'start': None, 'end': None, 'store': None, 'products': [{'name': 'CHILE Anaheim Kilo', 'brand': None, 'price': 49.9, 'sale_type': None, 'sale_description': 'a sólo $49.90', 'unit': 'Kilo', 'restrictions': 'máximo 5 kilos por cliente'}, {'name': 'CHILE Poblano Kilo', 'brand': None, 'price': 49.9, 'sale_type': None, 'sale_description': 'a sólo $49.90', 'unit': 'Kilo', 'restrictions': 'máximo 5 kilos por cliente'}, {'name': 'CHILE Serrano Kilo', 'brand': None, 'price': 49.9, 'sale_type': None, 'sale_description': 'a sólo $49.90', 'unit': 'Kilo', 'restrictions': 'máximo 5 kilos por cliente'}, {'name': 'CHILE Habanero Kilo', 'brand': None, 'price': 49.9, 'sale_type': None, 'sale_description': 'a sólo $49.90', 'unit': 'Kilo', 'restrictions': 'máximo 5 kilos por cliente'}, {'name': 'CHILE Jalapeño Kilo', 'brand': None, 'price': 49.9, 'sale_type': None, 'sale_description': 'a sólo $49.90', 'unit': 'Ki

In [ ]:
with open('qwen.json', 'w') as f:
    json.dump(flyer, f, indent= 4)

#### llama3.2

In [ ]:
batch_size = 1
flyer = {
    "products": [],
    "start_date": None,
    "end_date": None,
    "store": store,
    "legal_warnings": None,
    "extra_info": None
}

for i in range(0, len(imgs), batch_size):
    batch = imgs[i:i + batch_size]
    batch_num = i // batch_size + 1
    messages = [
            {
                'role': 'system',
                'content': encode(extract_text_prompt)
            },
            {
                'role': 'user',
                'content': f'Analiza las siguientes imágenes y extrae la información del supermercado {store}',
                'images': batch
            }
        ]
    
    try: 
        print(f"\n⏳ Enviando batch {batch_num} a Ollama (Contiene {len(batch)} imágenes)...")
        start_time = time.time()
        flyer_text = ollama.chat(
            model = "llama3.2-vision:11b",
            messages = messages,
            options = {'temperature': 0}
        )

        elapsed_time = time.time() - start_time
        print(f"✅ Ollama respondió en {elapsed_time:.2f} segundos.")

        batch_data  = clean_response(flyer_text.message.content)

        print(batch_data)

        if batch_data.get("products"):
            flyer["products"].extend(batch_data["products"])
            print(f"  Batch {batch_num}: {len(batch_data['products'])} productos")
            
        for key in ["start_date", "end_date", "legal_warnings", "extra_info"]:
            new_value = batch_data.get(key)
            
            if new_value and not flyer.get(key):
                flyer[key] = new_value
            
            elif new_value and flyer.get(key) and key in ["legal_warnings", "extra_info"]:
                if new_value not in flyer[key]:
                    flyer[key] = f"{flyer[key]} | {new_value}"
            
    except json.JSONDecodeError as e:
        print(f"✗ Error parseando JSON en batch {batch_num}: {e}")
    except ValidationError as e:
        print(f"✗ Error de validación en batch {batch_num}: {e}")
    except Exception as e:
        print(f"✗ Error procesando batch {batch_num}: {e}")

flyer['total_products'] = len(flyer['products'])

print(f"\nTotal de productos extraídos: {len(flyer['products'])}")


⏳ Enviando batch 1 a Ollama (Contiene 1 imágenes)...
✅ Ollama respondió en 26.22 segundos.
✗ Error parseando JSON en batch 1: Expecting value: line 1 column 1 (char 0)

⏳ Enviando batch 2 a Ollama (Contiene 1 imágenes)...


In [ ]:
with open('llama3_2.json', 'w') as f:
    json.dump(flyer, f, indent= 4)

### cloud models

In [9]:
from ollama import Client
from sina.config.credentials import ollama_api_key

client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + ollama_api_key}
)

#### prueba

In [10]:
response = client.chat(
    model="qwen3.5:397b-cloud",
    messages=[{
        'role': 'user',
        'content': '¿Qué ves en esta imagen?',
        'images': [str(imgs[0])]
    }]
)
print("Tokens:", response['prompt_eval_count'])
print(response['message']['content'][:500])

Tokens: 811
En esta imagen veo un **folleto promocional de un supermercado** que muestra ofertas en frutas y verduras frescas. La imagen está organizada en una cuadrícula con fotos de los productos, sus nombres y precios destacados en grande.

Aquí tienes el detalle de lo que aparece, fila por fila (de arriba a abajo, izquierda a derecha):

**Primera sección superior:**
*   **Chile Anaheim:** $49.90 el kilo.
*   **Mandarina:** $49.90 el kilo.
*   **Plátano Portalimón:** $27.50 el kilo.
*   **Chile Jalapeño:


In [12]:
print(response.message.content)

En esta imagen veo un **folleto promocional de un supermercado** que muestra ofertas en frutas y verduras frescas. La imagen está organizada en una cuadrícula con fotos de los productos, sus nombres y precios destacados en grande.

Aquí tienes el detalle de lo que aparece, fila por fila (de arriba a abajo, izquierda a derecha):

**Primera sección superior:**
*   **Chile Anaheim:** $49.90 el kilo.
*   **Mandarina:** $49.90 el kilo.
*   **Plátano Portalimón:** $27.50 el kilo.
*   **Chile Jalapeño:** $26.90 el kilo (ubicado debajo del plátano).

**Segunda fila:**
*   **Chile Serrano:** $39.90 el kilo.
*   **Cilantro:** $9.90 el mazo.
*   **Repollo:** $14.90 el kilo.

**Tercera fila:**
*   **Zanahoria (a granel):** $12.90 el kilo.
*   **Manzana Gala (en bolsa de 600g):** $39.50.
*   **Espinaca:** $9.90 el mazo.

**Cuarta fila:**
*   **Lechuga Romana:** $19.90 la pieza.
*   **Durazno Rojo:** $89.90 el kilo.
*   **Mango Haden:** $59.90 el kilo.

**Quinta fila:**
*   **Plátano Macho:** $39.90

#### c/ 1 imagen

In [10]:
messages = [
        {
            'role': 'system',
            'content': encode(extract_text_prompt)
        },
        {
            'role': 'user',
            'content': 'Analiza las siguientes imágenes y extrae la información',
            'images': [str(imgs[0])]
        }
    ]

In [11]:
start = time.perf_counter()

response = client.chat(
    model="qwen3.5:397b-cloud",
    messages=messages,
    options={'temperature': 0}
)

print(f"Duración: ", time.perf_counter() - start)

Duración:  27.52262210007757


In [12]:
dict(response)

{'model': 'qwen3.5:397b',
 'created_at': '2026-03-04T16:10:21.687718015Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 27055211129,
 'load_duration': None,
 'prompt_eval_count': 1629,
 'prompt_eval_duration': None,
 'eval_count': 2733,
 'eval_duration': None,
 'message': Message(role='assistant', content='{\n  "products": [\n    {\n      "name": "CHILE Anaheim",\n      "brand": null,\n      "price": 49.90,\n      "sale_type": "precio_directo",\n      "sale_description": "a sólo $49.90",\n      "unit": "Kilo",\n      "restrictions": "máximo 5 kilos por cliente"\n    },\n    {\n      "name": "MANDARINA",\n      "brand": null,\n      "price": 49.90,\n      "sale_type": "precio_directo",\n      "sale_description": "a sólo $49.90",\n      "unit": "Kilo",\n      "restrictions": "máximo 5 kilos por cliente"\n    },\n    {\n      "name": "PLÁTANO Portalimón",\n      "brand": null,\n      "price": 27.50,\n      "sale_type": "precio_directo",\n      "sale_description": "a sólo $27.

In [13]:
print(response.message.content)

{
  "products": [
    {
      "name": "CHILE Anaheim",
      "brand": null,
      "price": 49.90,
      "sale_type": "precio_directo",
      "sale_description": "a sólo $49.90",
      "unit": "Kilo",
      "restrictions": "máximo 5 kilos por cliente"
    },
    {
      "name": "MANDARINA",
      "brand": null,
      "price": 49.90,
      "sale_type": "precio_directo",
      "sale_description": "a sólo $49.90",
      "unit": "Kilo",
      "restrictions": "máximo 5 kilos por cliente"
    },
    {
      "name": "PLÁTANO Portalimón",
      "brand": null,
      "price": 27.50,
      "sale_type": "precio_directo",
      "sale_description": "a sólo $27.50",
      "unit": "Kilo",
      "restrictions": "máximo 5 kilos por cliente"
    },
    {
      "name": "CHILE Serrano",
      "brand": null,
      "price": 39.90,
      "sale_type": "precio_directo",
      "sale_description": "a sólo $39.90",
      "unit": "Kilo",
      "restrictions": "máximo 5 kilos por cliente"
    },
    {
      "name": "

#### c/ batches

In [14]:
batch_size = 1
flyer = {
    "products": [],
    "start_date": None,
    "end_date": None,
    "store": store,
    "legal_warnings": None,
    "extra_info": None
}

for i in range(0, len(imgs), batch_size):
    batch = imgs[i:i + batch_size]
    batch_num = i // batch_size + 1
    messages = [
            {
                'role': 'system',
                'content': encode(extract_text_prompt)
            },
            {
                'role': 'user',
                'content': f'Analiza las siguientes imágenes y extrae la información del supermercado {store}',
                'images': batch
            }
        ]
    
    try: 
        print(f"\n⏳ Enviando batch {batch_num} a Ollama (Contiene {len(batch)} imágenes)...")
        start_time = time.time()
        flyer_text = client.chat(
            model = "qwen3.5:397b-cloud",
            messages = messages,
            options = {'temperature': 0}
        )

        elapsed_time = time.time() - start_time
        print(f"✅ Ollama respondió en {elapsed_time:.2f} segundos.")

        batch_data  = clean_response(flyer_text.message.content)

        print(batch_data)

        if batch_data.get("products"):
            flyer["products"].extend(batch_data["products"])
            print(f"  Batch {batch_num}: {len(batch_data['products'])} productos")
            
        for key in ["start_date", "end_date", "legal_warnings", "extra_info"]:
            new_value = batch_data.get(key)
            
            if new_value and not flyer.get(key):
                flyer[key] = new_value
            
            elif new_value and flyer.get(key) and key in ["legal_warnings", "extra_info"]:
                if new_value not in flyer[key]:
                    flyer[key] = f"{flyer[key]} | {new_value}"
            
    except json.JSONDecodeError as e:
        print(f"✗ Error parseando JSON en batch {batch_num}: {e}")
    except ValidationError as e:
        print(f"✗ Error de validación en batch {batch_num}: {e}")
    except Exception as e:
        print(f"✗ Error procesando batch {batch_num}: {e}")

flyer['total_products'] = len(flyer['products'])

print(f"\nTotal de productos extraídos: {len(flyer['products'])}")


⏳ Enviando batch 1 a Ollama (Contiene 1 imágenes)...
✅ Ollama respondió en 52.36 segundos.
{'products': [{'name': 'CHILE Anaheim', 'brand': None, 'price': 49.9, 'sale_type': 'precio_directo', 'sale_description': 'a sólo $49.90', 'unit': 'Kilo', 'restrictions': '*máximo 5 kilos por cliente*'}, {'name': 'MANDARINA', 'brand': None, 'price': 49.9, 'sale_type': 'precio_directo', 'sale_description': 'a sólo $49.90', 'unit': 'Kilo', 'restrictions': '*máximo 5 kilos por cliente*'}, {'name': 'PLÁTANO Portalimón', 'brand': None, 'price': 27.5, 'sale_type': 'precio_directo', 'sale_description': 'a sólo $27.50', 'unit': 'Kilo', 'restrictions': '*máximo 5 kilos por cliente*'}, {'name': 'CHILE Serrano', 'brand': None, 'price': 39.9, 'sale_type': 'precio_directo', 'sale_description': 'a sólo $39.90', 'unit': 'Kilo', 'restrictions': '*máximo 5 kilos por cliente*'}, {'name': 'CILANTRO', 'brand': None, 'price': 9.9, 'sale_type': 'precio_directo', 'sale_description': 'a sólo $9.90', 'unit': 'Mazo', 'res

In [18]:
path = DATA / store / city / date / "flyer_data.json"
with open(path, 'w', encoding='utf-8') as f:
    json.dump(flyer, f, ensure_ascii=False, indent= 3)